# 14 — Final Summary and Conclusions

**FinRisk** — AI-Powered Investment Portfolio Risk Analytics, Optimization &
Risk Management Platform

> Educational and analytical purposes only. Risk estimates and portfolio
> recommendations are model-based and do not guarantee future performance.
> This is not investment advice.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

np.random.seed(42)

from notebooks.nb_utils import fetch_price_panel

TICKERS = ["AAPL", "MSFT", "GLD", "BND", "SPY"]
START = "2019-01-01"

price_panel, used_synthetic = fetch_price_panel(TICKERS, START)
returns_panel = price_panel.pct_change().dropna()
print(f"Loaded {price_panel.shape[0]} rows x {price_panel.shape[1]} tickers. Synthetic data: {used_synthetic}")
price_panel.tail()

Failed to get ticker 'SPY' reason: Expecting value: line 1 column 1 (char 0)


HTTP Error 403: Host not in allowlist: query2.finance.yahoo.com. Add this host to your network egress settings to allow access.


HTTP Error 403: Host not in allowlist: query1.finance.yahoo.com. Add this host to your network egress settings to allow access.


Failed to get ticker 'AAPL' reason: Expecting value: line 1 column 1 (char 0)


HTTP Error 403: Host not in allowlist: query2.finance.yahoo.com. Add this host to your network egress settings to allow access.


HTTP Error 403: Host not in allowlist: query1.finance.yahoo.com. Add this host to your network egress settings to allow access.


... (7 outputs left out)

['SPY', 'AAPL']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')


['BND', 'GLD', 'MSFT']: possibly delisted; no timezone found


[info] Live data unavailable (KeyError); using synthetic demo data instead.
Loaded 2004 rows x 5 tickers. Synthetic data: True


,AAPL,MSFT,GLD,BND,SPY
2026-08-31,229.681016,128.274867,119.914845,364.746718,107.728124
2026-09-01,227.093540,128.294744,120.000708,372.445526,106.711406
2026-09-02,229.155329,127.875189,121.085078,365.765614,106.991434
2026-09-03,232.958620,125.652809,122.697759,374.610608,107.656847
2026-09-04,232.705684,125.622649,120.900524,369.667262,106.661264


## What this project demonstrates

1. **Normalized, audit-friendly data architecture** — raw and cleaned
   prices are kept in separate MySQL tables (`database/schema.sql`), so a
   bug in cleaning can always be diagnosed against the untouched source.
2. **Leakage-safe feature and target construction by design, not by
   discipline** — every rolling feature is a trailing window; targets are
   built in a completely separate module (`src/models/targets.py`) from
   features; both are backed by automated tests (`tests/test_no_leakage.py`)
   that actively try to break the guarantees by perturbing future data --
   run `pytest tests/ -v` for the current, live count rather than a
   number written here, which would go stale the moment the suite changes
   (see `docs/SECOND_FORENSIC_AUDIT.md` Issue H12).
3. **A full risk toolkit** — historical/parametric/Monte Carlo VaR, CVaR,
   stress testing, and per-asset risk contribution, each with documented
   assumptions and limitations.
4. **Portfolio construction and rebalancing** — minimum-variance,
   maximum-Sharpe, and target-return optimization, plus a cost-aware
   rebalancing engine that only ever produces recommendations, never
   executes trades.
5. **A time-aware ML and backtesting pipeline** — chronological
   train/validation/test splitting with purge and embargo gaps, models
   wrapped in `Pipeline(preprocessing -> model)` so scalers never see
   validation/test data, and a walk-forward backtest engine with realistic
   execution lag and transaction costs.
6. **An interactive dashboard** (`streamlit run app/app.py`, 9 pages)
   putting all of the above in front of a user, with an explicit,
   user-chosen data source (MySQL or synthetic demo data) rather than a
   silent automatic fallback.

## Live test suite result

The count below is produced by actually running `pytest` from this cell, right now — never hard-coded. A hard-coded test count in a presentation notebook goes stale the moment the test suite changes, which happened repeatedly during this project's development; see `docs/SECOND_FORENSIC_AUDIT.md` Issue H12.

In [2]:
import subprocess

result = subprocess.run(
    ["pytest", str(PROJECT_ROOT / "tests"), "-q", "--no-header"],
    capture_output=True, text=True, cwd=str(PROJECT_ROOT),
)
summary_lines = [line for line in result.stdout.splitlines() if "passed" in line or "failed" in line or "error" in line]
print("\n".join(summary_lines) if summary_lines else result.stdout[-500:])

98 passed, 2 warnings in 15.32s


In [3]:
import json
from pathlib import Path

results_summary = {
    "risk_methods_implemented": ["historical VaR", "parametric VaR", "Monte Carlo VaR/CVaR",
                                   "historical CVaR", "stress testing", "risk contribution"],
    "optimization_methods": ["minimum variance", "maximum Sharpe", "target return", "efficient frontier"],
    "ml_models": ["logistic_regression", "random_forest", "hist_gradient_boosting"],
    "dashboard_pages": 9,
}
print(json.dumps(results_summary, indent=2))
print("\nFor the current test count, see the live pytest output in the cell above.")

{
  "risk_methods_implemented": [
    "historical VaR",
    "parametric VaR",
    "Monte Carlo VaR/CVaR",
    "historical CVaR",
    "stress testing",
    "risk contribution"
  ],
  "optimization_methods": [
    "minimum variance",
    "maximum Sharpe",
    "target return",
    "efficient frontier"
  ],
  "ml_models": [
    "logistic_regression",
    "random_forest",
    "hist_gradient_boosting"
  ],
  "dashboard_pages": 9
}

For the current test count, see the live pytest output in the cell above.


## Limitations (see README §20 for full detail)

- **Survivorship bias**: the data provider only returns currently-listed
  tickers; historical constituents that were delisted are missing and this
  is not corrected.
- **Small-sample ML results**: the demonstration model in notebooks 11–13
  is trained on a short history and/or synthetic data; real predictive
  value would need to be re-validated on a longer, real dataset before any
  production use.
- **VaR/CVaR are model-based estimates**, not guarantees of maximum
  possible loss — all three methods (historical, parametric, Monte Carlo)
  rely on the past being a reasonable guide to the near future.
- **Optimization uses sample covariance**, which is known to be noisy in
  high dimensions; no shrinkage estimator is applied yet.

## Future improvements

- Point-in-time index constituent data to remove survivorship bias.
- Shrinkage/EWMA covariance estimators as an alternative to sample covariance.
- SHAP-based interpretability for the risk-prediction model.
- Extending `fit_walk_forward_regimes` output directly into the backtest
  engine as a regime-conditional strategy switch.

## Disclaimer

This application is for educational and analytical purposes only. Risk
estimates and portfolio recommendations are model-based and do not
guarantee future performance. This is not investment advice.